# Stage 2 — decision-layer stabilisation mechanisms for continuous authentication (Colab)

Runs the complete, pre-registered experiment from a fresh runtime:
dataset download + MD5 check → audit → participant split → fixed score generator (Exp 1 gate) →
identity-transition benchmark → validation sweeps → frozen operating points → test evaluation →
statistics → tables, figures, report.

* Every stage is **checkpointed**. If the runtime disconnects, re-run the cells from the top: finished
  stages are skipped (optionally keep caches on Google Drive, cell 3).
* Nothing is fabricated: if the dataset cannot be downloaded or verified the run stops with a `DATA` failure.
* Runtime on a standard CPU runtime: pilot ≈ 2–3 min, primary run ≈ 10–20 min (download ≈ 1–2 min).

## 1. Get the code
Clones the repository branch. If you uploaded the repository ZIP instead, set `USE_ZIP = True` and put the ZIP in `/content`.

In [ ]:
USE_ZIP = False
REPO_URL = "https://github.com/TerryBinful/contectAware.git"
BRANCH = "experiment/stage2-mechanism-comparison"
import os, subprocess, glob
if USE_ZIP:
    z = sorted(glob.glob('/content/*.zip'))[0]
    subprocess.run(['unzip', '-oq', z, '-d', '/content'], check=True)
elif not os.path.exists('/content/contectAware'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, '/content/contectAware'], check=True)
STAGE2 = '/content/contectAware/experiment_Files/Stage2'
os.chdir(STAGE2)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
import sys; sys.path.insert(0, STAGE2 + '/src')
import numpy, pandas, scipy, sklearn, matplotlib
print({m.__name__: m.__version__ for m in (numpy, pandas, scipy, sklearn, matplotlib)})

## 3. Directories (optional: persist data and checkpoints on Google Drive)

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    base = '/content/drive/MyDrive/ca_stability'
else:
    base = '/content/ca_stability'
os.environ['CA_DATA_DIR'] = base + '/data'
os.environ['CA_WORK_DIR'] = base + '/work'
for d in (os.environ['CA_DATA_DIR'], os.environ['CA_WORK_DIR']):
    os.makedirs(d, exist_ok=True)
print(os.environ['CA_DATA_DIR'], os.environ['CA_WORK_DIR'])

## 4. Download and verify the dataset (public ExtraSensory archive, MD5-checked)

In [ ]:
!python scripts/download_data.py

## 5. Test-suite (mechanism definitions, batch/reference equivalence, metrics, leakage detection, synthetic end-to-end)

In [ ]:
!python -m pytest -q tests

## 6. Pilot run (8 genuine users) + reproducibility check
The pilot only checks the pipeline; its results are labelled PILOT and are not evidence.

In [ ]:
!python scripts/run_experiment.py --config configs/pilot.yaml
!python scripts/check_reproducibility.py --config configs/pilot.yaml

## 7. Primary run
Resumable: re-running skips completed stages. To recompute a stage: `--force <stage>`.

In [ ]:
!python scripts/run_experiment.py --config configs/main.yaml

## 8. Results

In [ ]:
import pandas as pd, json
from IPython.display import Markdown, Image, display
R = 'results/main'
print(json.dumps(json.load(open(f'{R}/baseline/exp1_summary.json'))['gate'], indent=1))
display(pd.read_csv(f'{R}/operating_points.csv').query("target_name == 'primary'")[['mechanism','params_str','in_band','val_false_locks_per_hour','val_ania_median','val_far_frame','best_smoother']])
for t in ['T5_security', 'T6_stability', 'T7_responsiveness', 'T8b_planned_contrasts']:
    display(Markdown(f'### {t}\n' + open(f'{R}/tables/{t}.md').read()))

In [ ]:
for f in ['fig03_frontier_val', 'fig04_outcomes_primary', 'fig05_tradeoffs', 'fig06_example_trace', 'fig09_critical_difference']:
    display(Image(f'{R}/figures/{f}.png'))

In [ ]:
display(Markdown(open(f'{R}/RESULTS_REPORT.md').read()))

## 9. Optional sensitivity analyses (secondary)

In [ ]:
# !python scripts/run_experiment.py --config configs/sens_motion_dynamics.yaml
# !python scripts/run_experiment.py --config configs/sens_original_features.yaml
# !python scripts/run_experiment.py --config configs/sens_all_participants.yaml

## 10. Download results

In [ ]:
!cd results && zip -qr /content/stage2_results.zip main pilot
from google.colab import files
files.download('/content/stage2_results.zip')